In [ ]:
# Imports
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import geopandas as gpd
import matplotlib.ticker as mticker
from matplotlib.colors import ListedColormap
import os

In [ ]:
# Paths and output directory
file_path = "../../Calabria_dataset/InputReteGood/Target/2017/20170701.h5"
save_dir = "plots/target_plots"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
# Inspect H5 file structure
with h5py.File(file_path, "r") as h5_file:
    print("Top-level keys:", list(h5_file.keys()))
    for key in h5_file.keys():
        print(f"'{key}' contains:", list(h5_file[key].keys()))

In [ ]:
# Extract grid attributes
with h5py.File(file_path, "r") as h5_file:
    attributes_table = h5_file["attributes/table"][:]

attribute_names = [attr[0].decode() for attr in attributes_table]
attribute_values = [attr[1][0] for attr in attributes_table]
attributes_dict = dict(zip(attribute_names, attribute_values))
attributes_dict

In [ ]:
# Extract ncols and nrows
ncols = int(attributes_dict["ncols"]) if "ncols" in attributes_dict else None
nrows = int(attributes_dict["nrows"]) if "nrows" in attributes_dict else None

print(f"Number of columns: {ncols}")
print(f"Number of rows: {nrows}")

In [ ]:
# Load wildfire occurrence values
with h5py.File(file_path, "r") as h5_file:
    values_table = h5_file["values/table"][:]

index_values = values_table["index"]
fire_results = values_table["values_block_0"].flatten()
fire_results

In [ ]:
# Convert index to row and column coordinates
row_coords = index_values // ncols
col_coords = index_values % ncols
row_coords, col_coords

In [ ]:
# Build DataFrame with spatial coordinates
df = pd.DataFrame({
    "Row": row_coords,
    "Column": col_coords,
    "Fire_Result": fire_results,
    "X_Coord": attributes_dict["xllcorner"] + (col_coords * attributes_dict["cellsize"]),
    "Y_Coord": attributes_dict["yllcorner"] + ((nrows - 1 - row_coords) * attributes_dict["cellsize"])
})
df

In [ ]:
# Round Y coordinates
df["Y_Coord"] = df["Y_Coord"].round(-0).astype(float).round(1)
df

In [ ]:
# Load precomputed cell-zone mapping
df_cell_zones = pd.read_parquet("../1_data/processed/cell_zones.parquet")
df_cell_zones

In [ ]:
# Load warning areas shapefile
gdf_zones = gpd.read_file("../1_data/raw/WarningAreas/WarningAreas.shp")
gdf_zones

In [ ]:
# Filter to only fire cells
df_fire_only = df[df["Fire_Result"] == 1].copy()
df_fire_only

In [ ]:
# Merge fire cells with zone coordinates
df_fire_with_coords = df_cell_zones[["X_Coord", "Y_Coord", "Zone_ID"]].merge(
    df_fire_only,
    on=["X_Coord", "Y_Coord"],
    how="inner"
)
df_fire_with_coords

In [ ]:
# Convert to GeoDataFrame
gdf_fire = gpd.GeoDataFrame(
    df_fire_with_coords,
    geometry=gpd.points_from_xy(df_fire_with_coords["X_Coord"], df_fire_with_coords["Y_Coord"]),
    crs="EPSG:32633"
)
gdf_fire

In [ ]:
# Count total wildfire cells for the day
num_fires = gdf_fire.shape[0]
print("Total wildfire cells:", num_fires)

In [ ]:
# Plot wildfire occurrence map
cmap = ListedColormap(plt.colormaps["Set1"].colors[:8])
gdf_zones_plot = gpd.GeoDataFrame(df_cell_zones,
                                  geometry=gpd.points_from_xy(df_cell_zones.X_Coord, df_cell_zones.Y_Coord),
                                  crs="EPSG:32633")

fig, ax = plt.subplots(figsize=(10, 10))
gdf_zones_plot.plot(column="Zone_ID", cmap=cmap, legend=True, ax=ax, markersize=30, alpha=0.5)
gdf_fire.plot(ax=ax, color="red", markersize=3, label="Wildfire Cells")
ax.set_title("Wildfire Occurrence Map", fontsize=14)
ax.set_xlabel("Easting (millions of meters)")
ax.set_ylabel("Northing (millions of meters)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x * 1e-6:.2f}M"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y * 1e-6:.2f}M"))
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "wildfire_occurrence_map.png"), dpi=300, bbox_inches='tight')